In [1]:
import tensorflow as tf
import numpy as np
import gym
import gym_mppt
from ou_noise import OUNoise
import random
from td3 import TD3


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })

def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.001
    CRITIC_LEARNING_RATE =  0.01
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v2'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
    epochs = 15000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000.
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = TD3(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0
            max_steps = 90 #es solo a efecto de contador
            r_episodio_actual = []
            llegadas =0
            while (not done):
                step += 1
                #print('step =', step)
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)

                action = np.clip(action,min_action,max_action)
                #print('LA ACCION clipeada ES', action, action.shape) 
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                #print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
                #print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                if step > max_steps:
                    llegadas +=1
                #print('done = ',done)
                #print ('--------------------------------------------')
                #print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
                #print ('--------------------------------------------')

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_TD301.npy',Reward_episodios)
 
            # print('FINNNNNN epoch = ',i)              
        
            print('epoch          = ', i)      
            print('Episode Reward = ', episode_r/5)
            print('Last Reward    = ', reward)
            print('State          = ', state)
            print()

            if i % 1000 == 0:      # Save model every 1000 iterations
                ddpg.save()

        ddpg.save()

C:\Users\archi\anaconda3\envs\BTP\lib\site-packages\gym\logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


epoch          =  0
Episode Reward =  2.7550019999999997
Last Reward    =  0.0
State          =  [   48.002 17971.473   623.83 ]

Model saved in file: actor_model
epoch          =  1
Episode Reward =  1.578564
Last Reward    =  0.0
State          =  [ 151.454   1847.1482  -272.46344]

epoch          =  2
Episode Reward =  1.6178880000000002
Last Reward    =  0.0
State          =  [  85.241   3124.3416   -58.86363]

epoch          =  3
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   43.372  16266.649   1205.2251]

epoch          =  4
Episode Reward =  2.9488600000000003
Last Reward    =  0.0
State          =  [   47.474  11711.8125 -1644.8345]

epoch          =  5
Episode Reward =  0.787282
Last Reward    =  0.0
State          =  [  207.043      0.     -6606.0034]

epoch          =  6
Episode Reward =  2.732999989894867
Last Reward    =  0.0
State          =  [ 139.446  3304.471   249.1814]

epoch          =  7
Episode Reward =  5.906892000000001
Last Reward    =  0.0


c:\users\archi\python code\btp\luis_upgrade\mppt-gym_v4\gym_mppt\envs\mppt_env_shaded2.py:39: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)


epoch          =  41
Episode Reward =  9.232166004959106
Last Reward    =  0.0
State          =  [206.706   0.      0.   ]

epoch          =  42
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  43
Episode Reward =  0.383614
Last Reward    =  0.0
State          =  [   44.659 10714.045 -1171.291]

epoch          =  44
Episode Reward =  2.56171
Last Reward    =  0.0
State          =  [   28.905  10850.535  -1325.0648]

epoch          =  45
Episode Reward =  1.3847640037536622
Last Reward    =  0.0
State          =  [   3.583   1345.3301   455.17557]

epoch          =  46
Episode Reward =  6.676297994926452
Last Reward    =  0.0
State          =  [ 107.77   3744.2756 -101.551 ]

epoch          =  47
Episode Reward =  7.18012198797989
Last Reward    =  0.0
State          =  [0. 0. 0.]

epoch          =  48
Episode Reward =  4.1937380000000015
Last Reward    =  0.0
State          =  [ 115.111   3854.4878   -17.71882]

epoch          =  49
Episode Re

epoch          =  105
Episode Reward =  5.354755990421295
Last Reward    =  0.0
State          =  [   17.691   6634.597  -1142.4456]

epoch          =  106
Episode Reward =  1.3853359942741394
Last Reward    =  0.0
State          =  [   93.145 34723.594  2254.207]

epoch          =  107
Episode Reward =  1.5862580060958862
Last Reward    =  0.0
State          =  [   97.848    16571.793      -75.078125]

epoch          =  108
Episode Reward =  4.697508000587463
Last Reward    =  0.0
State          =  [   46.445  17406.146   1645.7637]

epoch          =  109
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   74.145  27427.05   -1898.0059]

epoch          =  110
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   29.44   11050.569  -3300.0312]

epoch          =  111
Episode Reward =  2.3850899996490478
Last Reward    =  0.0
State          =  [   63.185  15157.329  -1258.3457]

epoch          =  112
Episode Reward =  3.486193999999999
Last Reward    =  0.0
Stat

...

epoch          =  12842
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   41.234  12410.82    -326.2207]

epoch          =  12843
Episode Reward =  49.45262590646744
Last Reward    =  0.0
State          =  [  109.885   27394.01      129.64453]

epoch          =  12844
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [  38.624  9537.427   906.8242]

epoch          =  12845
Episode Reward =  40.04533799886703
Last Reward    =  0.0
State          =  [   45.622  10789.093  -1071.3301]

epoch          =  12846
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   27.555  10197.971  -1755.9092]

epoch          =  12847
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   59.671  16128.562  -1803.2256]

epoch          =  12848
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   19.958   7478.1675 -2918.2075]

epoch          =  12849
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   17.731   6292.4146 -1626.381

epoch          =  12904
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [ 8.6375000e+01  2.9939186e+04 -5.5878906e+00]

epoch          =  12905
Episode Reward =  11.709172022342681
Last Reward    =  0.0
State          =  [   65.94    15592.816     555.51953]

epoch          =  12906
Episode Reward =  13.14764004945755
Last Reward    =  0.0
State          =  [   84.76    3108.553  -1655.8972]

epoch          =  12907
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   80.652 19805.6    2153.711]

epoch          =  12908
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   61.604  18513.885   2989.9658]

epoch          =  12909
Episode Reward =  1.9708580017089843
Last Reward    =  0.0
State          =  [   68.33  16302.113 -2178.209]

epoch          =  12910
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   78.397  18636.422   2218.3145]

epoch          =  12911
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   79.

epoch          =  12966
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [  102.327  30377.287    886.2383]

epoch          =  12967
Episode Reward =  49.664674079418184
Last Reward    =  0.9705299735069275
State          =  [   92.862   19209.795     304.85547]

epoch          =  12968
Episode Reward =  48.71413598060608
Last Reward    =  0.9770399928092957
State          =  [ 1.0271900e+02  2.2736986e+04 -1.5412109e+01]

epoch          =  12969
Episode Reward =  68.09004395008087
Last Reward    =  0.994159996509552
State          =  [  107.415   35511.5       -51.98828]

epoch          =  12970
Episode Reward =  70.45769795179368
Last Reward    =  0.0
State          =  [  124.083  44004.793  -1084.2148]

epoch          =  12971
Episode Reward =  48.27071197032929
Last Reward    =  0.9915500283241272
State          =  [  109.614  23133.098    415.9004]

epoch          =  12972
Episode Reward =  1.5504299998283386
Last Reward    =  0.0
State          =  [   50.07    8605.5

epoch          =  13027
Episode Reward =  18.36497994661331
Last Reward    =  0.0
State          =  [   79.013   17875.922    -173.56055]

epoch          =  13028
Episode Reward =  28.62670406103134
Last Reward    =  0.9663199782371521
State          =  [   81.104   28195.84      283.46094]

epoch          =  13029
Episode Reward =  38.769165980815885
Last Reward    =  0.0
State          =  [ 1.0507500e+02  2.5463857e+04 -1.4154297e+01]

epoch          =  13030
Episode Reward =  31.657323932647706
Last Reward    =  0.0
State          =  [   86.905   3178.46   -3983.6987]

epoch          =  13031
Episode Reward =  19.853908038139345
Last Reward    =  0.0
State          =  [   50.715  18173.945   -635.7383]

epoch          =  13032
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   53.659   18890.408     249.95508]

epoch          =  13033
Episode Reward =  0.0
Last Reward    =  0.0
State          =  [   31.263  11511.655    333.9795]

epoch          =  13034
Episode Rewar

KeyboardInterrupt: 